# Check the DRP stage2 (recalibrate) input and output datasets

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-30 : stage 2 ; restructured 2026-11-06 ; rewritten 2026-11-08 (inputs / outputs of stage 2, `fgcm_*` products)
- purpose : list and look at the datasets **read** (inputs) and **written** (outputs) by `DRP.yaml#stage2-recalibrate`, with a focus on the `fgcm_*` products of `fgcmBuildFromIsolatedStar`, `fgcmFitCycle` and `fgcmOutputProducts`
- the list of datasets comes from `notebooks/fig_diagrams/pipeline_drp_pipe_LSSTCam_DRP.yaml` and from the stage2 quantum graph `pipeline_drp_pipe_LSSTCam_DRP_stage2-recalibrate.png / .pdf` (same folder)

## Contents

1. Run description
2. Setup
3. Collections (stage1 = input, stage2 = output, run collections of stage 2)
4. Content of the stage2 collection : datasets written by stage 2
5. Pipeline steps (quantum diagrams) and tasks / datasets table
6. Expected datasets : inputs and outputs, found or missing
7. Input datasets : `isolated_star*`, `preliminary_visit_summary`, `fgcmLookUpTable`, `camera`, reference catalog
8. Output datasets of the fgcm tasks : intermediate products, fit cycles, atmosphere, zero points, final products, plots
9. Other output datasets : gbdes astrometry, turbulence, refit PSF, `visit_summary`, `recalibrated_star`, stellar motions, metrics
10. Stage 1 versus stage 2 : zero points of the visit summaries
11. Image and recalibrated catalog of one detector
12. Next notebooks

## 1. Run description

Submission of the 2026-11-03 run (`bps_dp2cosmos_2026-11-03_stage2.sh`, release `w_2026_43`, `bps_submit.sh bps_dp2cosmos_parsl_2026-11-03_stage2.yaml`) :

```bash
export PATCHIDS=$(paste -sd, patchIds.txt)
export TRACTIDS=$(paste -sd, tractIds.txt)
export CCDIDS=$(paste -sd, ccdIds.txt)
export NOVISITS=$(paste -sd, visits_to_exclude.txt)
```

- `file bps_dp2cosmos_parsl_2026-11-03_stage2.yaml` (abridged)

```
pipelineYaml: "${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage2-recalibrate"

payload:
  runInit: true
  payloadName: dp2cosmos202611
  butlerConfig: main
  inCollection: u/${USER}/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1,LSSTCam/defaults
  output: u/${USER}/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage2_v2
  dataQuery: "instrument='LSSTCam' AND skymap='lsst_cells_v2' AND tract IN (${TRACTIDS}) AND patch IN (${PATCHIDS}) AND detector IN (${CCDIDS}) AND exposure NOT IN (${NOVISITS}) AND exposure.observation_type='science'"

# fgcmFitCycle stopped with 'All observations in %s band have been cut' (u band : 202 of 206 visits TOO_FEW_STARS,
# 4 NO_STARS with the default minStarPerExp=500)
extraQgraphOptions: "-c fgcmFitCycle:minStarPerExp=100"
```

- the output collection of this submission is `.../stage2_v2`. The collection `.../stage2` is the output of the earlier `step2a-recalibrate-global` only submission (`bps_dp2cosmos_parsl_2026-11-03_step2a.yaml`, the fgcm tasks). The one explored is chosen with `STAGE2_NAME` in the configuration cell.
- the quantum graph for DRP.yaml#stage2-recalibrate: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage2-recalibrate.pdf
- all quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/

## 2. Setup

In [ ]:
import os
import re
import logging
import warnings
import importlib
from collections import Counter
from itertools import islice
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

import lsst.geom as geom
import lsst.pex.config as pexConfig
from lsst.daf.butler import Butler
from lsst.daf.butler.registry import MissingDatasetTypeError

import showplots_utils
importlib.reload(showplots_utils)   # pick up any edit of the helper module without restarting the kernel
from showplots_utils import show_diagram

In [ ]:
# Turn the Butler.collections FutureWarning into an error
warnings.filterwarnings(
    'error',
    category=FutureWarning,
    message='.*Butler.collections.*',
)

logging.basicConfig(
    level=logging.WARNING,
    format='%(name)s %(levelname)s: %(message)s',
    force=True,
)

butler = Butler('main')

In [ ]:
username = os.getenv('USER')
RUN_NAME = 'runcosmos01_20261103_tract9813patchsel_RunCosmos9patches'

# Stage-2 output collection :
#   'stage2_v2' = output of bps_dp2cosmos_parsl_2026-11-03_stage2.yaml (whole stage2, fgcmFitCycle:minStarPerExp=100)
#   'stage2'    = output of bps_dp2cosmos_parsl_2026-11-03_step2a.yaml (step2a only : the fgcm tasks)
STAGE2_NAME = 'stage2_v2'

stage1_collection = f'u/{username}/LSSTCam/{RUN_NAME}/stage1'            # input of stage 2
collection_path = f'u/{username}/LSSTCam/{RUN_NAME}/{STAGE2_NAME}'        # output of stage 2 (chained collection)
DEFAULTS_COLLECTION = 'LSSTCam/defaults'                                  # input of stage 2 : calibrations, refcats, skymap

NPRINT = 10              # number of dataIds printed per dataset type
COUNT_DATASETS = True    # count the datasets of each type (can take a few minutes : several hundred dataset types)

# Quantum diagrams of the pipeline steps
DIAGRAM_DIR = Path('../../../../notebooks/fig_diagrams')

FGCM_CYCLE = None        # fgcm cycle to look at (None = last cycle found in the stage2 outputs)
VISIT_ID = None          # visit used in the single-visit examples (None = first one available)
DETECTOR_ID = None       # detector used in the image + catalog example of section 11 (None = first one available)

print('user              :', username)
print('stage1 collection :', stage1_collection)
print('stage2 collection :', collection_path)

In [ ]:
def query_refs(dataset_type, collections, find_first=True, limit=None):
    '''Dataset references of a type in the collections ([] if the dataset type is not registered).

    find_first=True keeps one dataset per dataId (the first one found in the collection order).'''
    try:
        results = butler.registry.queryDatasets(dataset_type, collections=collections, findFirst=find_first)
        return list(islice(results, limit)) if limit else list(results)
    except MissingDatasetTypeError:
        return []


def count_refs(dataset_type, collections, find_first=True, exact=True):
    '''Number of datasets of a type (exact=False : 1 if there is at least one dataset, 0 otherwise).'''
    try:
        results = butler.registry.queryDatasets(dataset_type, collections=collections, findFirst=find_first)
        if not exact:
            return int(next(iter(results), None) is not None)
        try:
            return results.count()
        except Exception:
            return len(list(results))
    except MissingDatasetTypeError:
        return 0


def dval(ref, key):
    '''Value of a dimension in the dataId of a reference (None if absent).'''
    try:
        return ref.dataId[key]
    except Exception:
        return None


def visit_of(ref):
    '''Visit identifier of a reference : its visit, or its exposure (equal to the visit for LSSTCam).'''
    v = dval(ref, 'visit')
    return v if v is not None else dval(ref, 'exposure')


def pick_ref(refs, visit_id=None):
    '''Reference of the requested visit (the first one if visit_id is None, None if not found).'''
    for ref in refs:
        if visit_id is None or visit_of(ref) == visit_id:
            return ref
    return None


def to_pandas(table):
    '''Catalog (pandas / arrow / astropy / afw) to DataFrame ; array-valued columns are kept as object columns.'''
    if isinstance(table, pd.DataFrame):
        return table
    if hasattr(table, 'asAstropy'):          # afw catalogs
        table = table.asAstropy()
    if hasattr(table, 'to_pandas'):
        try:
            return table.to_pandas()
        except ValueError:                   # astropy : multidimensional columns
            data = {}
            for name in table.colnames:
                col = table[name]
                data[name] = list(np.asarray(col)) if col.ndim > 1 else np.asarray(col)
            return pd.DataFrame(data)
    raise TypeError(f'cannot convert {type(table)} to a DataFrame')

In [ ]:
def describe_table(table, nrows=5):
    '''Length, columns (with array shapes) and first rows of a catalog, whatever its python type.'''
    if hasattr(table, 'asAstropy'):          # afw catalogs
        table = table.asAstropy()
    n_rows = len(table)
    columns = getattr(table, 'colnames', None) or getattr(table, 'column_names', None) or list(table.columns)
    try:
        head = to_pandas(table[:nrows])
    except Exception:
        head = to_pandas(table).head(nrows)
    print(f'{n_rows} rows x {len(columns)} columns')
    print('columns :', list(columns))
    array_cols = {c: np.shape(head[c].iloc[0]) for c in head.columns
                  if len(head) and isinstance(head[c].iloc[0], np.ndarray)}
    if array_cols:
        print('array-valued columns (shape in the first row) :', array_cols)
    with pd.option_context('display.max_columns', 30, 'display.max_colwidth', 50, 'display.width', 250):
        display(head)
    return head


def describe(obj, nrows=5, max_config_fields=25):
    '''Describe what Butler.get returned, whatever the storage class (table, catalog, config, plot, mapping, ...).'''
    print('python type :', f'{type(obj).__module__}.{type(obj).__name__}')
    if isinstance(obj, plt.Figure):
        print('figure size (inches) :', obj.get_size_inches())
        display(obj)
        plt.close(obj)
    elif isinstance(obj, pexConfig.Config):
        fields = obj.toDict()
        print(f'config with {len(fields)} top-level fields (first {max_config_fields} shown)')
        for key, value in list(fields.items())[:max_config_fields]:
            text = repr(value)
            more = '...' if len(text) > 90 else ''
            print(f'  {key:40s} {text[:90]}{more}')
    elif isinstance(obj, pd.DataFrame) or hasattr(obj, 'asAstropy') or hasattr(obj, 'colnames') or hasattr(obj, 'to_pandas'):
        describe_table(obj, nrows)
    elif hasattr(obj, 'sampleAt'):           # afw TransmissionCurve
        print('wavelength bounds (Angstrom) :', obj.getWavelengthBounds())
    elif hasattr(obj, 'items'):              # dict-like : ArrowNumpyDict, MetricMeasurementBundle, ...
        entries = list(obj.items())
        print(f'mapping with {len(entries)} entries')
        for key, value in entries[:20]:
            shape = np.shape(value) if isinstance(value, (np.ndarray, list, tuple)) else ''
            print(f'  {str(key):45s} {type(value).__name__:20s} {shape}')
        if len(entries) > 20:
            print(f'  ... ({len(entries) - 20} more)')
    else:
        print(repr(obj)[:500])


def inspect_dataset(name, collections, find_first=True, limit=None, nprint=NPRINT):
    '''List the datasets of a type, load the first one and describe it. Returns (refs, object or None).'''
    display(Markdown(f'#### `{name}`'))
    refs = query_refs(name, collections, find_first, limit)
    if not refs:
        print('no dataset found (task not run or failed, or dataset type not registered)')
        return refs, None
    dt = refs[0].datasetType
    suffix = f' (limited to the first {limit})' if limit else ''
    print(f'dimensions : {list(dt.dimensions.names)} | storage class : {dt.storageClass_name}')
    print(f'{len(refs)} dataset(s){suffix}')
    for ref in refs[:nprint]:
        print('  ', ref.dataId)
    if len(refs) > nprint:
        print(f'   ... ({len(refs) - nprint} more)')
    try:
        obj = butler.get(refs[0])
    except Exception as exc:
        print(f'could not load the first dataset : {type(exc).__name__}: {str(exc)[:200]}')
        return refs, None
    try:
        describe(obj)
    except Exception as exc:
        print(f'could not describe the dataset : {type(exc).__name__}: {str(exc)[:200]}')
    return refs, obj


def show_plot(name, collections=None, index=0):
    '''Display one of the Plot datasets of a type (the figure is stored as a dataset of storage class Plot).'''
    refs = query_refs(name, collections if collections is not None else OUTPUT_COLLECTIONS)
    print(f'{name} : {len(refs)} dataset(s)')
    if refs:
        ref = refs[min(index, len(refs) - 1)]
        print('  ', ref.dataId)
        try:
            # Plot datasets are PNG files that the matplotlib formatter cannot read back as a Figure : show the file
            display(Image(data=butler.getURI(ref).read()))
        except Exception as exc:
            print(f'could not display the plot : {type(exc).__name__}: {str(exc)[:200]}')

## 3. Collections

The stage2 output collection written by BPS is a CHAINED collection : the RUN collection(s) written by the stage2 submission(s), followed by the input collections (stage1 and `LSSTCam/defaults`). Looking at the whole chain would mix the inputs with the outputs : the datasets **written by stage 2** are searched in the RUN collections located under the stage2 collection name.

In [ ]:
# Existing user collections
user_collections = sorted(butler.collections.query(f'u/{username}/*'))

for name in user_collections:
    print(name)

print(f'Number of collections : {len(user_collections)}')

In [ ]:
# Check the chosen collections (stops here if one does not exist)
for label, name in [('stage1 (input)', stage1_collection), ('stage2 (output)', collection_path)]:
    if name not in butler.collections.query(name):
        raise RuntimeError(f'Collection does not exist : {name} (see the list above, and STAGE2_NAME)')
    info = butler.collections.get_info(name)
    print(f'{label:16s} : {name} [{info.type}]')

# Structure of the stage2 chain
chain_info = butler.collections.get_info(collection_path)
print()
print('children of the stage2 collection :')
for child in list(getattr(chain_info, 'children', []))[:20]:
    print('  ', child)

flat = list(butler.collections.query(collection_path, flatten_chains=True))
stage2_runs = [c for c in flat if c.startswith(collection_path + '/')]
print()
print(f'{len(flat)} collections once flattened, of which {len(stage2_runs)} RUN collection(s) written by stage 2 :')
for run in stage2_runs:
    print('  ', run)

if not stage2_runs:
    print('WARNING : no run found under the stage2 collection name : the outputs below will be mixed with the inputs')
    stage2_runs = [collection_path]

OUTPUT_COLLECTIONS = stage2_runs                     # datasets written by stage 2
STAGE1 = [stage1_collection]                         # datasets read from stage 1
DEFAULTS = [DEFAULTS_COLLECTION]                     # calibrations, reference catalogs, skymap

## 4. Content of the stage2 collection : datasets written by stage 2

One row per dataset type written in the stage2 RUN collection(s) : number of datasets (one per dataId, the latest run first), dimensions, storage class and family. The dataset types are taken from the collection summaries, then counted type by type. If the summary is not available, all the datasets are scanned.

The fgcm plots (`fgcm_Cycle<N>_<name>_Plot`) make up most of the dataset types : one set for each fit cycle.

In [ ]:
FAMILIES = [
    ('task provenance (config, metadata, log)', '_(config|metadata|log)$|^packages$'),
    ('fgcm plots', '^fgcm_Cycle[0-9]+_.*_Plot$'),
    ('fgcm cycle products', '^fgcm_Cycle[0-9]+_'),
    ('fgcm other', '^fgcm'),
    ('gbdes astrometry', '^gbdes'),
    ('turbulence', 'turbulence'),
    ('refit PSF', '^refit_psf'),
    ('visit summary and tables', '^(visit_summary|visit_table|visit_detector_table)'),
    ('analysis plots', 'Plot$'),
    ('analysis metrics', 'metrics'),
    ('recalibrated stars', '^recalibrated_star'),
    ('association, stellar motion', '^isolated_star'),
]


def family(name):
    '''Family of a dataset type, from its name.'''
    for label, pattern in FAMILIES:
        if re.search(pattern, name):
            return label
    return 'other'


def dataset_types_in_collections(collections):
    '''Union of the dataset type names of the collection summaries (None if the summary is not available).'''
    names = set()
    try:
        for collection in collections:
            info = butler.collections.get_info(collection, include_summary=True)
            names.update(info.dataset_types)
    except Exception as exc:
        print('Collection summary not available :', exc)
        return None
    return sorted(names)


type_names = dataset_types_in_collections(OUTPUT_COLLECTIONS)

rows = []
if type_names is None:
    counter = Counter(r.datasetType.name for r in query_refs('*', OUTPUT_COLLECTIONS))
    for name, n in sorted(counter.items()):
        rows.append(dict(datasetType=name, n=n, dimensions='', storageClass='', family=family(name)))
else:
    for name in type_names:
        dt = butler.get_dataset_type(name)
        rows.append(dict(
            datasetType=name,
            n=count_refs(name, OUTPUT_COLLECTIONS) if COUNT_DATASETS else None,
            dimensions=', '.join(dt.dimensions.names),
            storageClass=dt.storageClass_name,
            family=family(name),
        ))

df_types = pd.DataFrame(rows).set_index('datasetType')

print(f'Stage2 run collection(s) : {OUTPUT_COLLECTIONS}')
print(f'Number of dataset types written by stage 2 : {len(df_types)}')
if COUNT_DATASETS:
    print(f'Total number of datasets : {int(df_types["n"].sum())}')

In [ ]:
# Summary by family
summary = df_types.groupby('family').agg(n_dataset_types=('family', 'size'), n_datasets=('n', 'sum'))
display(summary)

In [ ]:
# Filter the table by a string in the dataset type name (e.g. 'fgcm', 'gbdes', 'visit_', 'recalibrated', 'metrics')
NAME_FILTER = 'fgcm'
HIDE_PLOTS = True        # hide the Plot dataset types (hundreds of fgcm plots)

selection = df_types[df_types.index.str.contains(NAME_FILTER, case=False)]
if HIDE_PLOTS:
    selection = selection[~selection.index.str.endswith('Plot')]

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None, 'display.width', 250):
    display(selection)

## 5. Pipeline steps (quantum diagrams)

Tasks of `stage2-recalibrate` and their main datasets (compiled from `pipeline_drp_pipe_LSSTCam_DRP.yaml` and the stage2 quantum graph ; the exact dataset names are the ones checked in section 6) :

| Step | Task | Main inputs | Main outputs |
|---|---|---|---|
| 2a | `fgcmBuildFromIsolatedStar` | `isolated_star_association`, `isolated_star`, `preliminary_visit_summary`, `camera`, `fgcmLookUpTable`, `the_monster_20250219` | `fgcm_star_observations`, `fgcm_star_ids`, `fgcm_reference_stars`, `fgcmVisitCatalog` |
| 2a | `fgcmFitCycle` | the outputs of the previous task, `fgcmLookUpTable` | each cycle N : `fgcm_CycleN_OutputConfig`, `_FitParameters`, `_FlaggedStars` ; last cycle : `_Zeropoints`, `_AtmosphereParameters`, `_StandardStars` ; the `fgcm_CycleN_*_Plot` |
| 2a | `fgcmOutputProducts` | the products of the last fit cycle | `fgcmPhotoCalibCatalog`, `transmission_atmosphere_fgcm`, `fgcm_standard_star` |
| 2c | `gbdesHealpix3AstrometricFit` | `isolated_star_association`, `isolated_star`, `preliminary_visit_summary` | `gbdesHealpix3AstrometricFitSkyWcsCatalog`, `_fitStars`, `_starCatalog`, `_dcrCoefficients` |
| 2d | `gaussianProcessesTurbulenceFit` | the outputs of the gbdes fit | `turbulence_fit_hyperparameters`, `turbulenceCorrectedSkyWcsCatalog` |
| 2d | `refitPsfModelDetector`, `consolidateRefitPsfModelDetector` | `preliminary_visit_image`, `single_visit_star_footprints`, `isolated_star*` | `refit_psf_star_detector`, `refit_psf_models_detector`, then `refit_psf_star`, `refit_psf_models` |
| 2d | `updateVisitSummary` | `preliminary_visit_summary`, `preliminary_visit_image`, refit PSF, WCS (gbdes / turbulence), `fgcmPhotoCalibCatalog` | `visit_summary` |
| 2d | `recalibrateSingleVisitStar`, `standardizeRecalibratedStar`, `consolidateRecalibratedStar` | `visit_summary`, `single_visit_star_footprints` | `recalibrated_star_unstandardized`, `recalibrated_star_detector`, `recalibrated_star` |
| 2d | astrometric reference match analysis | `recalibrated_star`, `visit_summary` | `recalibrated_star_ref_match_astrom`, `_metrics`, `_metrics_table` |
| 2e | `analyzeRecalibratedStarAssociation`, `fitStellarMotion` | `recalibrated_star`, `isolated_star*`, `preliminary_visit_table` | `recalibrated_star_association_metrics` and its plots, `isolated_star_stellar_motions`, `isolated_star_predicted_positions` |
| 2f | `makeVisitTable`, `makeVisitDetectorTable`, metric tables, whole-sky plots | `visit_summary`, metrics | `visit_table`, `visit_detector_table`, `*_metrics_table`, `*_WholeSkyPlot` |

In [ ]:
DIAGRAM_STEPS = {
    'stage2-recalibrate':         'DRP stage2 : recalibrate (whole stage)',
    'step2a-recalibrate-global':  'DRP step2a/global (fgcm)',
    'step2c-recalibrate-healpix': 'DRP step2c/healpix (gbdes)',
    'step2d-recalibrate-visits':  'DRP step2d/visits',
}

# A missing file is reported and skipped
for step, label in DIAGRAM_STEPS.items():
    show_diagram(DIAGRAM_DIR / f'pipeline_drp_pipe_LSSTCam_DRP_{step}.png', title=label)

## 6. Expected datasets : inputs and outputs, found or missing

The dataset types read and written by stage 2 according to the quantum graph, and whether they are found in the collections. For the inputs that come from `LSSTCam/defaults` (a large collection) only the existence is tested (`n = 1` means *at least one*). A `MISSING` output means that the task did not run, failed, or wrote nothing (`--no-raise-on-partial-outputs`).

In [ ]:
WHERE = {'defaults': DEFAULTS, 'stage1': STAGE1, 'stage2': OUTPUT_COLLECTIONS}

STAGE2_INPUTS = [
    # (dataset type, dimensions, storage class, where, used by)
    ('camera', 'instrument', 'Camera', 'defaults', 'fgcmBuildFromIsolatedStar'),
    ('fgcmLookUpTable', 'instrument', 'Catalog', 'defaults', 'fgcmBuildFromIsolatedStar, fgcmFitCycle'),
    ('the_monster_20250219', 'htm7', 'SimpleCatalog', 'defaults', 'fgcmBuildFromIsolatedStar'),
    ('skyMap', 'skymap', 'SkyMap', 'defaults', 'tract-based tasks'),
    ('isolated_star_association', 'instrument, tract', 'ArrowAstropy', 'stage1',
     'fgcmBuildFromIsolatedStar, gbdesHealpix3AstrometricFit, refitPsfModelDetector, analyzeRecalibratedStarAssociation'),
    ('isolated_star', 'instrument, tract', 'ArrowAstropy', 'stage1',
     'fgcmBuildFromIsolatedStar, gbdesHealpix3AstrometricFit, refitPsfModelDetector, analyzeRecalibratedStarAssociation'),
    ('preliminary_visit_summary', 'visit', 'ExposureCatalog', 'stage1',
     'fgcmBuildFromIsolatedStar, gbdesHealpix3AstrometricFit, updateVisitSummary'),
    ('preliminary_visit_image', 'detector, visit', 'ExposureF', 'stage1', 'refitPsfModelDetector, updateVisitSummary'),
    ('single_visit_star_footprints', 'detector, visit', 'SourceCatalog', 'stage1',
     'refitPsfModelDetector, recalibrateSingleVisitStar'),
    ('preliminary_visit_table', 'instrument', 'ArrowAstropy', 'stage1', 'analyzeRecalibratedStarAssociation'),
]


def status_table(expected):
    '''One row per expected dataset type : is it there, and how many datasets ?'''
    rows = []
    for dataset_type, dims, storage_class, where, tasks in expected:
        n = count_refs(dataset_type, WHERE[where], exact=(where != 'defaults'))
        rows.append(dict(datasetType=dataset_type, dimensions=dims, storageClass=storage_class,
                         where=where, tasks=tasks, n=n, status='OK' if n > 0 else 'MISSING'))
    return pd.DataFrame(rows).set_index('datasetType')


df_inputs = status_table(STAGE2_INPUTS)

with pd.option_context('display.max_rows', None, 'display.max_colwidth', 70, 'display.width', 250):
    display(df_inputs)
print('missing inputs :', df_inputs.index[df_inputs['status'] == 'MISSING'].tolist())

In [ ]:
N_CYCLES_EXPECTED = 6    # fgcm_Cycle0 ... fgcm_Cycle5 in the quantum graph (fgcmFitCycle.doMultipleCycles)


def stage2_outputs(n_cycles=N_CYCLES_EXPECTED):
    '''Dataset types written by stage 2 (plots excluded), from the quantum graph.'''
    out = [
        # (dataset type, dimensions, storage class, task)
        ('fgcm_star_observations', 'instrument', 'ArrowAstropy', 'fgcmBuildFromIsolatedStar'),
        ('fgcm_star_ids', 'instrument', 'ArrowAstropy', 'fgcmBuildFromIsolatedStar'),
        ('fgcm_reference_stars', 'instrument', 'ArrowAstropy', 'fgcmBuildFromIsolatedStar'),
        ('fgcmVisitCatalog', 'instrument', 'Catalog', 'fgcmBuildFromIsolatedStar'),
    ]
    for c in range(n_cycles):
        for suffix, storage_class in (('OutputConfig', 'Config'), ('FitParameters', 'Catalog'), ('FlaggedStars', 'Catalog')):
            out.append((f'fgcm_Cycle{c}_{suffix}', 'instrument', storage_class, 'fgcmFitCycle'))
    last = n_cycles - 1
    for suffix, storage_class in (('Zeropoints', 'Catalog'), ('AtmosphereParameters', 'Catalog'), ('StandardStars', 'SimpleCatalog')):
        out.append((f'fgcm_Cycle{last}_{suffix}', 'instrument', storage_class, 'fgcmFitCycle'))
    out += [
        ('fgcmPhotoCalibCatalog', 'visit', 'ExposureCatalog', 'fgcmOutputProducts'),
        ('transmission_atmosphere_fgcm', 'visit', 'TransmissionCurve', 'fgcmOutputProducts'),
        ('fgcm_standard_star', 'instrument, tract', 'ArrowAstropy', 'fgcmOutputProducts'),
        ('gbdesHealpix3AstrometricFitSkyWcsCatalog', 'healpix3, visit', 'ExposureCatalog', 'gbdesHealpix3AstrometricFit'),
        ('gbdesHealpix3AstrometricFit_fitStars', 'healpix3, physical_filter', 'ArrowNumpyDict', 'gbdesHealpix3AstrometricFit'),
        ('gbdesHealpix3AstrometricFit_starCatalog', 'healpix3, physical_filter', 'ArrowNumpyDict', 'gbdesHealpix3AstrometricFit'),
        ('gbdesHealpix3AstrometricFit_dcrCoefficients', 'healpix3, physical_filter', 'ArrowNumpyDict', 'gbdesHealpix3AstrometricFit'),
        ('turbulence_fit_hyperparameters', 'healpix3, visit', 'ArrowAstropy', 'gaussianProcessesTurbulenceFit'),
        ('turbulenceCorrectedSkyWcsCatalog', 'healpix3, visit', 'ExposureCatalog', 'gaussianProcessesTurbulenceFit'),
        ('refit_psf_star_detector', 'detector, visit', 'ArrowAstropy', 'refitPsfModelDetector'),
        ('refit_psf_models_detector', 'detector, visit', 'ExposureCatalog', 'refitPsfModelDetector'),
        ('refit_psf_star', 'visit', 'ArrowAstropy', 'consolidateRefitPsfModelDetector'),
        ('refit_psf_models', 'visit', 'ExposureCatalog', 'consolidateRefitPsfModelDetector'),
        ('visit_summary', 'visit', 'ExposureCatalog', 'updateVisitSummary'),
        ('recalibrated_star_unstandardized', 'detector, visit', 'ArrowAstropy', 'recalibrateSingleVisitStar'),
        ('recalibrated_star_detector', 'detector, visit', 'ArrowAstropy', 'standardizeRecalibratedStar'),
        ('recalibrated_star', 'visit', 'ArrowAstropy', 'consolidateRecalibratedStar'),
        ('visit_detector_table', 'instrument', 'ArrowAstropy', 'makeVisitDetectorTable'),
        ('visit_table', 'instrument', 'ArrowAstropy', 'makeVisitTable'),
        ('recalibrated_star_ref_match_astrom', 'visit', 'ArrowAstropy', 'makeAnalysisRecalibratedStarAstrometricRefMatch'),
        ('recalibrated_star_ref_match_astrom_metrics', 'visit', 'MetricMeasurementBundle', 'analyzeRecalibratedStarAstrometricRefMatch'),
        ('recalibrated_star_ref_match_astrom_metrics_table', 'instrument', 'ArrowAstropy', 'makeRecalibratedStarAstrometricRefMatchMetricTable'),
        ('isolated_star_stellar_motions', 'instrument, tract', 'ArrowAstropy', 'fitStellarMotion'),
        ('isolated_star_predicted_positions', 'instrument, tract', 'ArrowAstropy', 'fitStellarMotion'),
        ('recalibrated_star_association_metrics', 'instrument, tract', 'MetricMeasurementBundle', 'analyzeRecalibratedStarAssociation'),
        ('recalibrated_star_association_metrics_table', 'skymap', 'ArrowAstropy', 'makeAnalysisRecalibratedStarAssociationMetricTable'),
    ]
    return [(name, dims, sc, 'stage2', task) for name, dims, sc, task in out]


df_outputs = status_table(stage2_outputs())

with pd.option_context('display.max_rows', None, 'display.max_colwidth', 60, 'display.width', 250):
    display(df_outputs[['dimensions', 'storageClass', 'tasks', 'n', 'status']])

print('status by task :')
display(df_outputs.groupby('tasks')['status'].value_counts().unstack(fill_value=0))
print('missing outputs :', df_outputs.index[df_outputs['status'] == 'MISSING'].tolist())

## 7. Input datasets

What stage 2 reads. The stage1 products are looked up in the stage1 collection (one dataset per dataId, the latest run first) and the calibrations in `LSSTCam/defaults`.

### 7.1 `isolated_star_association` and `isolated_star`

Product of `associateIsolatedStar` (stage 1 step 1c) : the isolated stars found by matching the `single_visit_star` catalogs. `isolated_star_association` has one row per isolated star, `isolated_star` one row per source of these stars. They are the star lists of `fgcmBuildFromIsolatedStar`, `gbdesHealpix3AstrometricFit` and of the repeatability analysis.

In [ ]:
refs_assoc, assoc = inspect_dataset('isolated_star_association', STAGE1)
refs_iso, iso = inspect_dataset('isolated_star', STAGE1)

if iso is not None:
    df_iso = to_pandas(iso)
    for column in ('band', 'physical_filter'):
        if column in df_iso.columns:
            display(df_iso[column].value_counts().rename(f'isolated_star sources per {column}').to_frame())

### 7.2 `preliminary_visit_summary`

Per-visit summary (one row per detector : zero point, PSF size, sky background, ...) written by `consolidateVisitSummary` at stage 1. It is the input of `fgcmBuildFromIsolatedStar` (the visit catalog is built from it), of the gbdes fit and of `updateVisitSummary`, which produces `visit_summary`.

In [ ]:
refs_pre, pre_summary = inspect_dataset('preliminary_visit_summary', STAGE1, nprint=5)
print('visits per band :', dict(Counter(dval(r, 'band') for r in refs_pre)))

### 7.3 `fgcmLookUpTable`, `camera`, `the_monster_20250219`

- `fgcmLookUpTable` : the atmospheric look-up table of fgcm (computed with MODTRAN)
- `camera` : the camera geometry (detectors)
- `the_monster_20250219` : the reference catalog (htm7 shards) used by fgcm for the absolute calibration. Only the first shards are listed.

In [ ]:
refs_lut, lut = inspect_dataset('fgcmLookUpTable', DEFAULTS)

display(Markdown('#### `camera`'))
visit0 = visit_of(refs_pre[0]) if refs_pre else None
try:
    camera = butler.get('camera', instrument='LSSTCam', exposure=visit0, collections=DEFAULTS)
    print(f'camera {camera.getName()} : {len(camera)} detectors (calibration valid for exposure {visit0})')
except Exception as exc:
    print(f'could not load the camera : {type(exc).__name__}: {str(exc)[:200]}')

refs_monster, monster = inspect_dataset('the_monster_20250219', DEFAULTS, limit=5)

## 8. Output datasets of the fgcm tasks

The three fgcm tasks of step 2a : `fgcmBuildFromIsolatedStar` (builds the star and visit lists), `fgcmFitCycle` (the fit, one set of products per cycle) and `fgcmOutputProducts` (per-visit photometric calibration and atmosphere). Which products exist depends on how far `fgcmFitCycle` went : see the tables of section 6 and 8.2.

The tables of sections 4 and 6 are the starting point : `df_types` (counts per dataset type) is used in 8.2.

### 8.1 Products of `fgcmBuildFromIsolatedStar`

- `fgcm_star_ids`, `fgcm_star_observations` : the stars and their observations selected for the fit
- `fgcm_reference_stars` : the reference catalog stars matched to them
- `fgcmVisitCatalog` : one row per visit (the visits kept for the fit)

In [ ]:
for name in ['fgcm_star_ids', 'fgcm_star_observations', 'fgcm_reference_stars']:
    inspect_dataset(name, OUTPUT_COLLECTIONS)

refs_vcat, visit_catalog = inspect_dataset('fgcmVisitCatalog', OUTPUT_COLLECTIONS)

if visit_catalog is not None:
    df_vcat = to_pandas(visit_catalog)
    for column in ('band', 'physicalFilter', 'filtername'):
        if column in df_vcat.columns:
            display(df_vcat[column].value_counts().rename(f'fgcmVisitCatalog : visits per {column}').to_frame())

### 8.2 Products of `fgcmFitCycle` : the fit cycles

Table of the `fgcm_Cycle<N>_*` dataset types found in the stage2 outputs, one row per cycle (number of datasets, number of plot dataset types and datasets). `FGCM_CYCLE` in the configuration cell selects the cycle examined below (default : the last one found).

In [ ]:
fgcm_rows = []
for name in df_types.index:
    match = re.match('^fgcm_Cycle([0-9]+)_(.+)$', name)
    if match:
        n = df_types.loc[name, 'n'] if COUNT_DATASETS else 1
        fgcm_rows.append(dict(cycle=int(match.group(1)), product=match.group(2), n=n))

if fgcm_rows:
    df_fgcm = pd.DataFrame(fgcm_rows)
    is_plot = df_fgcm['product'].str.endswith('_Plot')
    per_cycle = df_fgcm[~is_plot].pivot(index='cycle', columns='product', values='n')
    per_cycle['n_plot_types'] = df_fgcm[is_plot].groupby('cycle').size()
    per_cycle['n_plots'] = df_fgcm[is_plot].groupby('cycle')['n'].sum()
    display(per_cycle)
    fgcm_cycles = sorted(int(c) for c in per_cycle.index)
else:
    fgcm_cycles = []
    print('no fgcm_Cycle* dataset in the stage2 outputs : fgcmFitCycle did not write anything')

cycle = FGCM_CYCLE if FGCM_CYCLE is not None else (fgcm_cycles[-1] if fgcm_cycles else None)
print('fgcm cycles found :', fgcm_cycles, '| cycle examined below :', cycle)

#### Configuration of the fit : `fgcm_Cycle<N>_OutputConfig`

The `FgcmFitCycleConfig` actually used by the cycle (including the `-c fgcmFitCycle:minStarPerExp=100` override of this run).

In [ ]:
KEY_CONFIG_FIELDS = ['cycleNumber', 'bands', 'minStarPerExp', 'minCcdPerExp', 'maxIterBeforeFinalCycle', 'nCore',
                     'doReferenceCalibration']

if cycle is not None:
    refs_cfg, cfg = inspect_dataset(f'fgcm_Cycle{cycle}_OutputConfig', OUTPUT_COLLECTIONS)
    if isinstance(cfg, pexConfig.Config):
        fields = cfg.toDict()
        print()
        print('selected fields :')
        for key in KEY_CONFIG_FIELDS:
            value = fields.get(key, 'n/a')
            print(f'  {key:28s} {value}')
else:
    print('no fgcm cycle available')

#### Fit parameters and flagged stars : `fgcm_Cycle<N>_FitParameters`, `fgcm_Cycle<N>_FlaggedStars`

In [ ]:
if cycle is not None:
    refs_fit, fit_parameters = inspect_dataset(f'fgcm_Cycle{cycle}_FitParameters', OUTPUT_COLLECTIONS)
    refs_flag, flagged_stars = inspect_dataset(f'fgcm_Cycle{cycle}_FlaggedStars', OUTPUT_COLLECTIONS)

### 8.3 Products of the last cycle : zero points, atmosphere, standard stars

- `fgcm_Cycle<N>_Zeropoints` : the fgcm zero point of each (visit, detector)
- `fgcm_Cycle<N>_AtmosphereParameters` : the atmosphere fitted for each visit (precipitable water vapour, aerosols, ozone)
- `fgcm_Cycle<N>_StandardStars` : the fgcm standard star catalog

These datasets are only written by the last cycle : if `cycle` is not the last one of the run, they are reported as missing.

In [ ]:
zpt = atm = std = None
if cycle is not None:
    refs_zpt, zpt = inspect_dataset(f'fgcm_Cycle{cycle}_Zeropoints', OUTPUT_COLLECTIONS)
    refs_atm, atm = inspect_dataset(f'fgcm_Cycle{cycle}_AtmosphereParameters', OUTPUT_COLLECTIONS)
    refs_std, std = inspect_dataset(f'fgcm_Cycle{cycle}_StandardStars', OUTPUT_COLLECTIONS)

Distribution of the fgcm zero points per band, and time series of the atmosphere parameters. The column names are looked for in the catalogs (printed above) : adapt them if the schema is different.

In [ ]:
# Zero points
if zpt is not None:
    df_zpt = to_pandas(zpt)
    if 'fgcmFlag' in df_zpt.columns:
        print('fgcmFlag values :', df_zpt['fgcmFlag'].value_counts().to_dict())
    if 'fgcmZpt' in df_zpt.columns:
        group_col = next((c for c in ('filtername', 'band', 'physical_filter') if c in df_zpt.columns), None)
        zp = df_zpt['fgcmZpt'].to_numpy(dtype=float)
        good = np.isfinite(zp) & (zp > 0)
        print(f'{int(good.sum())} finite positive zero points out of {len(zp)}')
        fig, ax = plt.subplots(figsize=(8, 5))
        if group_col is None:
            ax.hist(zp[good], bins=60)
        else:
            for key, sub in df_zpt[good].groupby(group_col):
                ax.hist(sub['fgcmZpt'], bins=60, histtype='step', label=str(key))
            ax.legend(title=group_col)
        ax.set_xlabel('fgcmZpt (mag)')
        ax.set_ylabel('number of (visit, detector)')
        ax.set_title(f'fgcm_Cycle{cycle}_Zeropoints')
        plt.show()
    else:
        print('column fgcmZpt not found : see the columns printed above')
else:
    print('no zero point catalog available')

In [ ]:
# Atmosphere parameters : precipitable water vapour, aerosol optical depth, aerosol index, ozone
if atm is not None:
    df_atm = to_pandas(atm)
    params = [c for c in ('pwv', 'tau', 'alpha', 'o3') if c in df_atm.columns]
    print('atmosphere parameters found :', params)
    if params:
        x = df_atm['visit'] if 'visit' in df_atm.columns else df_atm.index
        fig, axes = plt.subplots(len(params), 1, figsize=(10, 2.6 * len(params)), sharex=True)
        axes = np.atleast_1d(axes)
        for ax, param in zip(axes, params):
            ax.plot(x, df_atm[param], '.', ms=3)
            ax.set_ylabel(param)
            ax.grid(alpha=0.3)
        axes[-1].set_xlabel('visit' if 'visit' in df_atm.columns else 'row')
        axes[0].set_title(f'fgcm_Cycle{cycle}_AtmosphereParameters')
        plt.show()
else:
    print('no atmosphere parameter catalog available')

### 8.4 Products of `fgcmOutputProducts` : final photometric calibration

- `fgcmPhotoCalibCatalog` : the `PhotoCalib` of each detector, per visit (read by `updateVisitSummary`)
- `transmission_atmosphere_fgcm` : the atmospheric transmission curve of each visit
- `fgcm_standard_star` : the standard star catalogs, per tract

In [ ]:
refs_pc, photo_calib_cat = inspect_dataset('fgcmPhotoCalibCatalog', OUTPUT_COLLECTIONS, nprint=5)
refs_tc, transmission = inspect_dataset('transmission_atmosphere_fgcm', OUTPUT_COLLECTIONS, nprint=5)
refs_fstd, fgcm_std_star = inspect_dataset('fgcm_standard_star', OUTPUT_COLLECTIONS, nprint=5)

In [ ]:
# PhotoCalib of the detectors of one visit : calibration mean (nJy per ADU) versus detector
ref_pc = pick_ref(refs_pc, VISIT_ID)
if ref_pc is not None:
    cat_pc = butler.get(ref_pc)
    pairs = [(rec['id'], rec.getPhotoCalib().getCalibrationMean()) for rec in cat_pc if rec.getPhotoCalib() is not None]
    if pairs:
        det, mean = np.array(pairs).T
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(det, mean, 'o', ms=4)
        ax.set_xlabel('detector')
        ax.set_ylabel('calibration mean (nJy / ADU)')
        ax.set_title(f'fgcmPhotoCalibCatalog, visit {visit_of(ref_pc)} : {len(pairs)} detectors')
        ax.grid(alpha=0.3)
        plt.show()
    else:
        print('no PhotoCalib in this catalog')
else:
    print('no fgcmPhotoCalibCatalog available')

In [ ]:
# Atmospheric transmission of the first visits (at the focal plane centre)
N_CURVES = 5

if refs_tc:
    fig, ax = plt.subplots(figsize=(10, 5))
    for ref in refs_tc[:N_CURVES]:
        try:
            curve = butler.get(ref)
            lo, hi = curve.getWavelengthBounds()
            wavelengths = np.linspace(lo, hi, 800)          # Angstrom
            ax.plot(wavelengths / 10.0, curve.sampleAt(geom.Point2D(0, 0), wavelengths), label=f'visit {visit_of(ref)}')
        except Exception as exc:
            print(f'visit {visit_of(ref)} : {type(exc).__name__}: {str(exc)[:150]}')
    ax.set_xlabel('wavelength (nm)')
    ax.set_ylabel('atmospheric transmission')
    ax.set_title('transmission_atmosphere_fgcm')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()
else:
    print('no transmission_atmosphere_fgcm available')

### 8.5 Fitted parameters versus time

The scalar quantities fitted by fgcm, plotted against `MJD - MJD_min` (bottom axis) with the calendar date on the top axis. They are read from the catalogs of the selected cycle (not from the PNG plots of 8.6) so that they can be selected and combined :

- `NightlyAlpha`, `NightlyO3`, `NightlyPwv`, `NightlyTau` : columns `alpha`, `o3`, `pwv`, `tau` of `fgcm_Cycle<N>_AtmosphereParameters` (one row per visit)
- `Zeropoints` : column `fgcmZpt` of `fgcm_Cycle<N>_Zeropoints`, one panel per band (small dots : each detector ; red : median over the detectors of a visit)

The MJD of a visit comes from `fgcmVisitCatalog` (or from the stage1 `preliminary_visit_table`). Choose the quantities in the selection cell ; the other names of the list of fgcm plots (`AbsThroughputs`, `Expgray`, ...) can be added to the `FGCM_QUANTITIES` dictionary once their source column is known.

In [ ]:
import matplotlib.dates as mdates

MJD_UNIX_EPOCH = 40587.0     # MJD of 1970-01-01 (epoch of the matplotlib dates)
BAND_ORDER = ['u', 'g', 'r', 'i', 'z', 'y']


def find_col(df, candidates):
    '''Name of the first column of df matching one of the candidates (case-insensitive), or None.'''
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def clean_band(values):
    '''Band names as plain strings (bytes decoded, physical filter names like g_01 reduced to g).'''
    s = pd.Series(list(values)).map(lambda v: v.decode() if isinstance(v, bytes) else str(v)).str.strip()
    return s.str.split('_').str[0].to_numpy()


def build_visit_table():
    '''DataFrame (visit, mjd, band) : from fgcmVisitCatalog, else from the stage1 preliminary_visit_table.'''
    for collections, name in [(OUTPUT_COLLECTIONS, 'fgcmVisitCatalog'), (STAGE1, 'preliminary_visit_table')]:
        refs = query_refs(name, collections)
        if not refs:
            continue
        df = to_pandas(butler.get(refs[0]))
        c_visit = find_col(df, ['visit', 'visitId'])
        c_mjd = find_col(df, ['mjd', 'expMidptMJD', 'obsStartMJD'])
        c_band = find_col(df, ['band', 'physicalFilter', 'physical_filter', 'filtername'])
        if c_visit and c_mjd:
            out = pd.DataFrame({'visit': df[c_visit].to_numpy(np.int64), 'mjd': df[c_mjd].to_numpy(float)})
            out['band'] = clean_band(df[c_band]) if c_band else ''
            out = out.drop_duplicates('visit')
            print(f'visit times from {name} : {len(out)} visits')
            return out
        print(f'{name} : no visit / mjd column found in {df.columns.tolist()}')
    return None


def row_times(df):
    '''(visit, mjd, band) of each row of a per-visit catalog : from the catalog itself if it has the columns,
    otherwise from the visit table.'''
    c_visit = find_col(df, ['visit', 'visitId', 'exposure'])
    c_mjd = find_col(df, ['mjd'])
    c_band = find_col(df, ['band', 'filtername', 'physical_filter'])
    out = pd.DataFrame(index=range(len(df)))
    out['visit'] = df[c_visit].to_numpy(np.int64) if c_visit else -1
    if c_mjd:
        out['mjd'] = df[c_mjd].to_numpy(float)
    if c_band:
        out['band'] = clean_band(df[c_band])
    if df_visit_times is not None and c_visit:
        lookup = df_visit_times.set_index('visit')
        if 'mjd' not in out:
            out['mjd'] = out['visit'].map(lookup['mjd'])
        if 'band' not in out:
            out['band'] = out['visit'].map(lookup['band'])
    return out


def add_date_axis(ax, mjd0):
    '''Top axis with readable dates (YYYY-MM-DD, tilted) for an axis whose x is MJD - mjd0.'''
    secax = ax.secondary_xaxis('top', functions=(lambda x: x + mjd0 - MJD_UNIX_EPOCH,
                                                 lambda d: d - mjd0 + MJD_UNIX_EPOCH))
    secax.xaxis.set_major_locator(mdates.AutoDateLocator())
    secax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    secax.tick_params(axis='x', labelrotation=45)
    ax.figure.canvas.draw()
    for label in secax.get_xticklabels():
        label.set_horizontalalignment('left')
    return secax


df_visit_times = build_visit_table()
if df_visit_times is not None:
    print(f'MJD from {df_visit_times["mjd"].min():.3f} to {df_visit_times["mjd"].max():.3f}')
else:
    print('no visit table with an MJD found : the MJD must then be a column of the plotted catalogs')

In [ ]:
# Quantities that can be plotted versus time.   name : (source, column of the catalog, axis label)
#   source 'atm' : fgcm_Cycle<N>_AtmosphereParameters (one row per visit)
#   source 'zpt' : fgcm_Cycle<N>_Zeropoints (one row per visit and detector, plotted per band)
FGCM_QUANTITIES = {
    'NightlyAlpha': ('atm', 'alpha', 'aerosol index alpha'),
    'NightlyO3':    ('atm', 'o3', 'ozone (Dobson units)'),
    'NightlyPwv':   ('atm', 'pwv', 'PWV (mm)'),
    'NightlyTau':   ('atm', 'tau', 'aerosol optical depth tau'),
    'Zeropoints':   ('zpt', 'fgcmZpt', 'fgcm zero point (mag)'),
}


def plot_atmosphere(names, catalog):
    '''Atmosphere parameters (one panel per name) versus MJD - MJD_min, calendar dates on the top axis.'''
    if not names:
        return
    if catalog is None:
        print('no AtmosphereParameters catalog available')
        return
    df = to_pandas(catalog)
    times = row_times(df)
    if 'mjd' not in times or not np.isfinite(times['mjd']).any():
        print('no MJD for the rows of the catalog ; columns :', df.columns.tolist())
        return
    mjd0 = float(np.nanmin(times['mjd']))
    x = times['mjd'].to_numpy(float) - mjd0
    fig, axes = plt.subplots(len(names), 1, figsize=(11, 2.8 * len(names)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, names):
        column, label = FGCM_QUANTITIES[name][1:]
        if column in df.columns:
            ax.plot(x, df[column].to_numpy(float), '.', ms=3)
        else:
            ax.text(0.5, 0.5, f'column {column} not found', transform=ax.transAxes, ha='center')
            print(f'{name} : column {column} not in', df.columns.tolist())
        ax.set_ylabel(label)
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel(f'MJD - {mjd0:.3f}  (days)')
    add_date_axis(axes[0], mjd0)
    fig.suptitle(f'fgcm_Cycle{cycle}_AtmosphereParameters', y=1.04)
    plt.show()


def plot_zeropoints(catalog, require_flag=None):
    '''fgcm zero points versus MJD - MJD_min, one panel per band, calendar dates on the top axis.'''
    if catalog is None:
        print('no Zeropoints catalog available')
        return
    df = to_pandas(catalog)
    column = FGCM_QUANTITIES['Zeropoints'][1]
    if column not in df.columns:
        print(f'column {column} not in', df.columns.tolist())
        return
    d = row_times(df)
    if 'mjd' not in d or 'band' not in d:
        print('no MJD or band for the rows of the catalog ; columns :', df.columns.tolist())
        return
    d['zpt'] = df[column].to_numpy(float)
    keep = (np.isfinite(d['zpt']) & (d['zpt'] > 0) & np.isfinite(d['mjd'])).to_numpy()
    if require_flag is not None and 'fgcmFlag' in df.columns:
        keep &= (df['fgcmFlag'].to_numpy() == require_flag)
    d = d[keep]
    print(f'{len(d)} zero points kept out of {len(df)}')
    if not len(d):
        return
    present = set(d['band'])
    bands = [b for b in BAND_ORDER if b in present] + sorted(present - set(BAND_ORDER))
    mjd0 = float(d['mjd'].min())
    fig, axes = plt.subplots(len(bands), 1, figsize=(11, 2.6 * len(bands)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, band in zip(axes, bands):
        sub = d[d['band'] == band]
        ax.plot(sub['mjd'] - mjd0, sub['zpt'], '.', ms=1.5, alpha=0.25, color='C0')
        med = sub.groupby('visit').agg(mjd=('mjd', 'first'), zpt=('zpt', 'median'))
        ax.plot(med['mjd'] - mjd0, med['zpt'], 'o', ms=3, color='C3')
        ax.set_ylabel(f'{band} : zero point (mag)')
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel(f'MJD - {mjd0:.3f}  (days)')
    add_date_axis(axes[0], mjd0)
    fig.suptitle(f'fgcm_Cycle{cycle}_Zeropoints (red : median over the detectors of a visit)', y=1.04)
    plt.show()

In [ ]:
# Choose here what to plot (names of FGCM_QUANTITIES)
QUANTITIES_TO_PLOT = ['NightlyAlpha', 'NightlyO3', 'NightlyPwv', 'NightlyTau', 'Zeropoints']

# fgcmFlag value to keep for the zero points (see the fgcmFlag counts printed in 8.3) ; None = no cut
ZPT_REQUIRE_FLAG = None

unknown = [n for n in QUANTITIES_TO_PLOT if n not in FGCM_QUANTITIES]
if unknown:
    print('unknown quantities :', unknown, '| available :', list(FGCM_QUANTITIES))
selected = [n for n in QUANTITIES_TO_PLOT if n in FGCM_QUANTITIES]

plot_atmosphere([n for n in selected if FGCM_QUANTITIES[n][0] == 'atm'], atm)
if 'Zeropoints' in selected:
    plot_zeropoints(zpt, require_flag=ZPT_REQUIRE_FLAG)

### 8.6 fgcm plots

The `fgcm_Cycle<N>_<name>_Plot` datasets (storage class `Plot`) of the selected cycle. Edit `FGCM_PLOT_NAMES` to see others : the list of the plots written by the cycle is printed first.

In [ ]:
FGCM_PLOT_NAMES = ['Zeropoints', 'NightlyPwv', 'NightlyTau', 'ExpgrayInitial', 'SigmaCal']

if cycle is not None:
    plot_types = [n for n in df_types.index if re.match(f'^fgcm_Cycle{cycle}_.*_Plot$', n)]
    print(f'{len(plot_types)} plot dataset types in fgcm_Cycle{cycle} :')
    print([n.replace(f'fgcm_Cycle{cycle}_', '').replace('_Plot', '') for n in plot_types])
    print()
    for plot_name in FGCM_PLOT_NAMES:
        show_plot(f'fgcm_Cycle{cycle}_{plot_name}_Plot')
else:
    print('no fgcm cycle available')

## 9. Other output datasets

### 9.1 Astrometry : `gbdesHealpix3AstrometricFit*` and turbulence

In [ ]:
for name in ['gbdesHealpix3AstrometricFitSkyWcsCatalog',
             'gbdesHealpix3AstrometricFit_fitStars',
             'gbdesHealpix3AstrometricFit_starCatalog',
             'gbdesHealpix3AstrometricFit_dcrCoefficients',
             'turbulence_fit_hyperparameters',
             'turbulenceCorrectedSkyWcsCatalog']:
    inspect_dataset(name, OUTPUT_COLLECTIONS, nprint=5)

### 9.2 PSF : `refit_psf_*`

In [ ]:
for name in ['refit_psf_star_detector', 'refit_psf_models_detector', 'refit_psf_star', 'refit_psf_models']:
    inspect_dataset(name, OUTPUT_COLLECTIONS, nprint=5)

### 9.3 Final visit summary and tables : `visit_summary`, `visit_table`, `visit_detector_table`

In [ ]:
for name in ['visit_summary', 'visit_table', 'visit_detector_table']:
    inspect_dataset(name, OUTPUT_COLLECTIONS, nprint=5)

### 9.4 Recalibrated stars : `recalibrated_star*`

In [ ]:
for name in ['recalibrated_star_unstandardized', 'recalibrated_star_detector', 'recalibrated_star']:
    inspect_dataset(name, OUTPUT_COLLECTIONS, nprint=5)

### 9.5 Stellar motions, metrics and analysis plots

In [ ]:
for name in ['isolated_star_stellar_motions',
             'isolated_star_predicted_positions',
             'recalibrated_star_ref_match_astrom',
             'recalibrated_star_ref_match_astrom_metrics',
             'recalibrated_star_ref_match_astrom_metrics_table',
             'recalibrated_star_association_metrics',
             'recalibrated_star_association_metrics_table']:
    inspect_dataset(name, OUTPUT_COLLECTIONS, nprint=5)

show_plot('recalibrated_star_association_stellarPhotometricRepeatability_HistPlot')
show_plot('recalibrated_star_association_stellarAstrometricRepeatability1_HistPlot')

## 10. Stage 1 versus stage 2 : zero points of the visit summaries

`preliminary_visit_summary` (input, stage 1) and `visit_summary` (output, stage 2) have the same rows (one per detector) : the zero point of `visit_summary` comes from the fgcm photometric calibration. The median over the detectors of the zero point difference, for each visit, shows what the recalibration changed.

In [ ]:
MAX_VISITS = None        # limit the number of visits compared (None = all)

refs_pre_by_visit = {visit_of(r): r for r in query_refs('preliminary_visit_summary', STAGE1)}
refs_fin_by_visit = {visit_of(r): r for r in query_refs('visit_summary', OUTPUT_COLLECTIONS)}
common_visits = sorted(refs_pre_by_visit.keys() & refs_fin_by_visit.keys())
print(f'preliminary_visit_summary : {len(refs_pre_by_visit)}, visit_summary : {len(refs_fin_by_visit)}, in common : {len(common_visits)}')

rows = []
failures = []
for visit in common_visits[:MAX_VISITS]:
    try:
        pre = to_pandas(butler.get(refs_pre_by_visit[visit]))[['id', 'zeroPoint']]
        fin = to_pandas(butler.get(refs_fin_by_visit[visit]))[['id', 'zeroPoint']]
        both = pre.merge(fin, on='id', suffixes=('_pre', '_final'))
        delta = both['zeroPoint_final'] - both['zeroPoint_pre']
        rows.append(dict(visit=visit, band=dval(refs_pre_by_visit[visit], 'band'), n_detectors=len(both),
                         median_dzp=np.nanmedian(delta), std_dzp=np.nanstd(delta)))
    except Exception as exc:
        failures.append((visit, f'{type(exc).__name__}: {str(exc)[:120]}'))

df_dzp = pd.DataFrame(rows)
print(f'{len(df_dzp)} visits compared, {len(failures)} failures')
if failures:
    print('first failure :', failures[0])

In [ ]:
if len(df_dzp):
    display(df_dzp.groupby('band')['median_dzp'].describe())

    fig, ax = plt.subplots(figsize=(10, 5))
    for band, sub in df_dzp.groupby('band'):
        ax.plot(sub['visit'], sub['median_dzp'], 'o', ms=4, label=str(band))
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('visit')
    ax.set_ylabel('median over detectors of delta zeroPoint (mag)')
    ax.set_title('zeroPoint(visit_summary) - zeroPoint(preliminary_visit_summary)')
    ax.legend(title='band')
    ax.grid(alpha=0.3)
    plt.show()
else:
    print('no visit compared')

## 11. Image and recalibrated catalog of one detector

The image is the stage1 `preliminary_visit_image` (input of stage 2) and the catalog the stage2 `recalibrated_star_detector` (output). Recalibration changes the WCS and the photometric calibration, not the pixel positions : the stars of the catalog must still sit on the sources of the image.

In [ ]:
refs_img = query_refs('preliminary_visit_image', STAGE1)
CATALOG_NAME = 'recalibrated_star_detector'
refs_cat = query_refs(CATALOG_NAME, OUTPUT_COLLECTIONS)

if not refs_cat:
    # stage 2 did not write the recalibrated catalogs (updateVisitSummary not run) : use the stage1 catalogs instead
    CATALOG_NAME = 'single_visit_star_detector'
    refs_cat = query_refs(CATALOG_NAME, STAGE1)
    print('recalibrated_star_detector not found : falling back on single_visit_star_detector (stage1)')

if not refs_img or not refs_cat:
    raise RuntimeError(f'nothing to show : {len(refs_img)} images, {len(refs_cat)} catalogs')

print('image dimensions   :', list(refs_img[0].dataId.dimensions.names))
print('catalog dimensions :', list(refs_cat[0].dataId.dimensions.names))

img_by_key = {(visit_of(r), dval(r, 'detector')): r for r in refs_img}
cat_by_key = {(visit_of(r), dval(r, 'detector')): r for r in refs_cat}

common_keys = sorted(img_by_key.keys() & cat_by_key.keys())

print(f'images : {len(img_by_key)}, catalogs : {len(cat_by_key)}, common (visit, detector) pairs : {len(common_keys)}')
print('visits    :', sorted({k[0] for k in common_keys}))
print('detectors :', sorted({k[1] for k in common_keys}))

In [ ]:
# Selection : VISIT_ID / DETECTOR_ID of the configuration cell (None = first available)
candidates = [
    k for k in common_keys
    if (VISIT_ID is None or k[0] == VISIT_ID) and (DETECTOR_ID is None or k[1] == DETECTOR_ID)
]
if not candidates:
    raise RuntimeError(f'No common (visit, detector) pair for VISIT_ID={VISIT_ID}, DETECTOR_ID={DETECTOR_ID}')

key = candidates[0]
ref_img, ref_cat = img_by_key[key], cat_by_key[key]

exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)
df_cat = to_pandas(catalog)

print('visit, detector  :', key)
print('image   dataId   :', ref_img.dataId)
print('catalog dataId   :', ref_cat.dataId)
print('image type       :', type(exposure))
print('image shape      :', exposure.image.array.shape)
print('catalog type     :', type(catalog))
print('catalog length   :', len(df_cat))
print('catalog columns  :', df_cat.columns.tolist())

Display of the image, with the catalog stars overlaid when pixel centroid columns are found. Check that the circles sit on sources, otherwise read the column list above and adapt `find_xy_columns`.

In [ ]:
def find_xy_columns(df):
    '''Names of the pixel centroid columns of a catalog, or None.'''
    names = {c.lower(): c for c in df.columns}
    for cx, cy in [('x', 'y'),
                   ('centroid_x', 'centroid_y'),
                   ('slot_centroid_x', 'slot_centroid_y'),
                   ('base_sdsscentroid_x', 'base_sdsscentroid_y'),
                   ('xcentroid', 'ycentroid')]:
        if cx in names and cy in names:
            return names[cx], names[cy]
    return None


img = exposure.image.array
bbox = exposure.getBBox()
extent = (bbox.getMinX() - 0.5, bbox.getMaxX() + 0.5, bbox.getMinY() - 0.5, bbox.getMaxY() + 0.5)
vmin, vmax = np.nanpercentile(img, [5, 99])

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(img, origin='lower', cmap='gray', vmin=vmin, vmax=vmax, extent=extent)
ax.set_xlabel('x (pixel)')
ax.set_ylabel('y (pixel)')

cols = find_xy_columns(df_cat)
if cols is not None:
    ax.scatter(df_cat[cols[0]], df_cat[cols[1]], s=40, facecolors='none', edgecolors='r', linewidths=0.6)
    ax.set_title(f'visit {key[0]}, detector {key[1]} : {len(df_cat)} stars of {CATALOG_NAME} ({cols[0]}, {cols[1]})')
else:
    ax.set_title(f'visit {key[0]}, detector {key[1]} : no x, y centroid columns found in the catalog')
    print('No x, y centroid columns found : see the list of catalog columns above')

plt.show()

## 12. Next notebooks

- `nb2026-11-06_ShowPlots_DRPstage2-recalibrate.ipynb` : the `lsst.analysis.tools` plots of this collection
- `nb2026-11-05_PlotPreliminaryImage_DRPstage2-recalibrate.ipynb` : the preliminary visit images
- `nb2026-11-05_CheckCollections_DRPstage1-single-visit.ipynb` : same check for stage 1 (input of this stage)